In [1]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo
import importlib.resources
import sys
import os
import json

In [3]:
root_path = os.path.join(os.getcwd(), "..", "Data", "fossil_results")
# read generator parameters and names
gen_path = gen_path = os.path.join(os.getcwd(), "..", "Data", "gen_dict.json")
with open(gen_path, 'rb') as f:
    gen_dict = json.load(f)
gen_names = list(gen_dict["fossil"].keys())

In [13]:
gen_csv_path = os.path.join(root_path, "gen_" + gen_names[0] + "_result.csv")
df_gen = pd.read_csv(gen_csv_path)
df_gen

,Time,LMP,power_to_grid,gen_101_CT_1.op_mode,gen_101_CT_1.startup,gen_101_CT_1.shutdown,gen_101_CT_1.power,elec_revenue,total_hourly_cost,total_hourly_revenue,hourly_startup_cost,net_hourly_cash_inflow,gen_101_CT_1.vom
0,1,0.000000,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0.000000,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.000000,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.000000,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.000000,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8779,8780,26.324557,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8780,8781,26.324557,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8781,8782,76.324557,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8782,8783,75.908700,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
# get column_info
def get_col_ts_info(df, col_name):
    return df[col_name]

def summarize_gen_pt_result(df, gen_name):
    tot_elec_rev = get_col_ts_info(df_gen, 'elec_revenue').sum()
    tot_op_cost = get_col_ts_info(df_gen, 'total_hourly_cost').sum()
    tot_startup_cost = get_col_ts_info(df_gen, "hourly_startup_cost").sum()
    tot_profit = get_col_ts_info(df_gen, "net_hourly_cash_inflow").sum()
    num_startup = get_col_ts_info(df_gen, f"gen_{gen_name}.startup").sum()
    
    sum_dict = {"tot_elec_rev": [tot_elec_rev],
               "tot_op_cost": [tot_op_cost],
               "tot_startup_cost": [tot_startup_cost],
               "tot_profit": [tot_profit],
               "num_startup": [int(num_startup)],}
    df_summarize = pd.DataFrame(sum_dict)
    df_summarize.index = [gen_name]
    
    return df_summarize

summarize_gen_pt_result(df_gen, gen_names[0])

,tot_elec_rev,tot_op_cost,tot_startup_cost,tot_profit,num_startup
101_CT_1,4.407069e+06,394224.227869,2328.615,4.010517e+06,45


In [21]:
df_all_gen_summarize = pd.DataFrame()
for name in gen_names:
    gen_csv_path = os.path.join(root_path, "gen_" + name + "_result.csv")
    df_gen = pd.read_csv(gen_csv_path)
    df_summarize = summarize_gen_pt_result(df_gen, name)
    df_all_gen_summarize = pd.concat([df_all_gen_summarize, df_summarize], ignore_index=True)

In [22]:
df_all_gen_summarize

,tot_elec_rev,tot_op_cost,tot_startup_cost,tot_profit,num_startup
0,4.407069e+06,3.942242e+05,2.328615e+03,4.010517e+06,45
1,4.407069e+06,3.942242e+05,2.328615e+03,4.010517e+06,45
2,2.800817e+07,8.271056e+06,6.197809e+05,1.911733e+07,58
3,2.800817e+07,8.271056e+06,6.197809e+05,1.911733e+07,58
4,4.401271e+06,4.057930e+05,2.276868e+03,3.993202e+06,44
...,...,...,...,...,...
67,1.037976e+08,1.509489e+07,2.916855e+06,8.578583e+07,104
68,1.507211e+07,1.925696e+06,5.608582e+05,1.258556e+07,99
69,1.507211e+07,1.925696e+06,5.608582e+05,1.258556e+07,99
70,1.043119e+08,1.528252e+07,3.141228e+06,8.588816e+07,112
